# 06 When Machine Learning Performs Better: Nonlinear Choice Data

## Program Overview

This teaching program extends Notebook 05 by changing the data-generating
process. In Notebook 05, the simulated choices were close to a linear utility
model, so the traditional Logit model performed very well. In this notebook,
the simulated choices are driven by threshold rules and variable interactions.
That makes the data less suitable for a single linear utility equation and more
favorable to tree-based machine learning models.

## Connection to Part 3 PPT

This notebook still begins from random utility and model validation, but it
shows a case where prediction-oriented models can outperform a traditional
linear Logit specification. It is useful after discussing maximum likelihood,
model comparison, and the difference between behavioral interpretation and
prediction.

## Overall Task AI Prompt (English)

Use this notebook to explain why model performance depends on the structure of
the data. Compare a traditional linear Logit model with Decision Tree and
Random Forest classifiers. For each output, explain whether the result supports
behavioral interpretation, predictive accuracy, or both.

## Learning Objectives

- Recognize nonlinear and threshold-based choice patterns.
- Explain why a linear Logit model may underfit nonlinear data.
- Compare traditional and machine learning models using test-set metrics.
- Interpret tree-based feature importance carefully.
- State when machine learning is useful without claiming it is always better.

## AI Prompt for This Code Cell

Generate nonlinear mode-choice data. Explain the threshold rules that make this data more favorable to tree-based machine learning models.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2027)
n = 1000

time_diff = rng.normal(0, 12, n)
cost_diff = rng.normal(1.5, 3.2, n)
reliability_diff = rng.normal(0, 8, n)
transfers = rng.integers(0, 4, n)
income_high = rng.binomial(1, 0.45, n)

good_fast = (time_diff < -7) & (cost_diff < 3)
good_cheap = (cost_diff < -1.2) & (transfers <= 1)
income_fast = income_high & (time_diff < 2) & (cost_diff < 4)
bad_reliability = (time_diff > 6) & (reliability_diff > 5)

score = (
    -1.10
    + 2.60 * good_fast
    + 2.10 * good_cheap
    + 1.40 * income_fast
    - 2.20 * bad_reliability
    - 1.40 * (transfers >= 2)
    - 1.20 * (cost_diff > 4.5)
)

prob_car = 1.0 / (1.0 + np.exp(-score))
choice_car = rng.binomial(1, prob_car)

data = pd.DataFrame(
    {
        "time_diff": time_diff,
        "cost_diff": cost_diff,
        "reliability_diff": reliability_diff,
        "transfers": transfers,
        "income_high": income_high,
        "choice_car": choice_car,
    }
)

rule_summary = pd.DataFrame(
    {
        "rule": ["good_fast", "good_cheap", "bad_reliability"],
        "observations": [
            int(good_fast.sum()),
            int(good_cheap.sum()),
            int(bad_reliability.sum()),
        ],
        "car_choice_rate": [
            choice_car[good_fast].mean(),
            choice_car[good_cheap].mean(),
            choice_car[bad_reliability].mean(),
        ],
    }
)

summary = pd.Series(
    {
        "observations": len(data),
        "car_choices": int(data["choice_car"].sum()),
        "transit_choices": int((1 - data["choice_car"]).sum()),
        "car_choice_rate": data["choice_car"].mean(),
    }
)

print(summary)
rule_summary.round(3)

observations       1000.000
car_choices         299.000
transit_choices     701.000
car_choice_rate       0.299
dtype: float64


,rule,observations,car_choice_rate
0,good_fast,189,0.778
1,good_cheap,92,0.783
2,bad_reliability,82,0.024


## Output Interpretation

The data set has 1,000 observations, with 299 car choices and 701 transit choices, so the overall car choice rate is 29.9%. The rule summary shows why this data is nonlinear. Under the `good_fast` rule, 189 observations have a car choice rate of about 77.8%; under the `good_cheap` rule, 92 observations have a car choice rate of about 78.3%; under the `bad_reliability` rule, 82 observations have a car choice rate of only about 2.4%. These are threshold patterns, not smooth one-unit linear effects. A traditional Logit model can estimate average directions, but tree-based models are better suited to finding rules such as "if car is much faster and not too costly, probability increases sharply.

## AI Prompt for This Code Cell

Split the nonlinear data into training and test sets, then estimate a traditional linear Logit model. Interpret its coefficients as average linear effects.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

features = [
    "time_diff",
    "cost_diff",
    "reliability_diff",
    "transfers",
    "income_high",
]

train, test = train_test_split(
    data,
    test_size=0.30,
    random_state=5,
    stratify=data["choice_car"],
)

traditional_logit = LogisticRegression(max_iter=1000)
traditional_logit.fit(train[features], train["choice_car"])

logit_coefficients = pd.Series(
    traditional_logit.coef_[0],
    index=features,
    name="coefficient",
)

print(f"Training observations: {len(train)}")
print(f"Test observations: {len(test)}")
print(f"Intercept: {traditional_logit.intercept_[0]:.4f}")
print(logit_coefficients.round(4))

Training observations: 700
Test observations: 300
Intercept: 0.3897
time_diff          -0.0753
cost_diff          -0.3097
reliability_diff   -0.0182
transfers          -0.8236
income_high         0.0813
Name: coefficient, dtype: float64


## Output Interpretation

The split creates 700 training observations and 300 test observations. The traditional Logit estimates an intercept of 0.3897. The coefficients are `time_diff = -0.0753`, `cost_diff = -0.3097`, `reliability_diff = -0.0182`, `transfers = -0.8236`, and `income_high = 0.0813`. These signs are behaviorally reasonable: larger car time, cost, reliability disadvantage, and transfers reduce car choice utility, while high income increases it. However, these are average linear effects. They cannot fully represent the threshold rules in the data, such as the sharp jump when `time_diff < -7` and `cost_diff < 3`, or the sharp penalty when reliability is bad.

## AI Prompt for This Code Cell

Train Decision Tree and Random Forest models. These models can split the feature space and are therefore better suited to threshold-based choice patterns.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

decision_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=20,
    random_state=5,
)

random_forest = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=8,
    random_state=5,
)

decision_tree.fit(train[features], train["choice_car"])
random_forest.fit(train[features], train["choice_car"])

print("Machine learning models trained on nonlinear choice data.")
print(f"Decision Tree depth: {decision_tree.get_depth()}")
print(f"Random Forest trees: {len(random_forest.estimators_)}")

Machine learning models trained on nonlinear choice data.
Decision Tree depth: 5
Random Forest trees: 300


## Output Interpretation

The Decision Tree reaches depth 5, and the Random Forest trains 300 trees. These details matter for interpretation. The single tree uses a limited set of hierarchical split rules, while the forest averages many trees to reduce variance and produce more stable predicted probabilities. Because the data were generated with threshold rules, these models can represent the structure more naturally than a single linear utility equation. This does not make them causal models; it means they are better aligned with the prediction task in this synthetic nonlinear setting.

## AI Prompt for This Code Cell

Compare the traditional Logit, Decision Tree, and Random Forest on held-out test data using multiple predictive metrics.

In [4]:
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

models = {
    "Traditional linear Logit": traditional_logit,
    "Decision Tree": decision_tree,
    "Random Forest": random_forest,
}

rows = []
for model_name, model in models.items():
    probability = model.predict_proba(test[features])[:, 1]
    prediction = model.predict(test[features])
    rows.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(test["choice_car"], prediction),
            "log_loss": log_loss(test["choice_car"], probability),
            "brier_score": brier_score_loss(test["choice_car"], probability),
            "roc_auc": roc_auc_score(test["choice_car"], probability),
        }
    )

comparison = pd.DataFrame(rows)
comparison.round(4)

,model,accuracy,log_loss,brier_score,roc_auc
0,Traditional linear Logit,0.7733,0.4542,0.1520,0.8255
1,Decision Tree,0.8267,0.8129,0.1255,0.8775
2,Random Forest,0.8233,0.3841,0.1222,0.8871


## Output Interpretation

The test-set results show the intended contrast with Notebook 05. The traditional Logit reaches accuracy 77.33%, log-loss 0.4542, Brier score 0.1520, and ROC-AUC 0.8255. The Decision Tree has the highest accuracy at 82.67%, improving classification by 5.34 percentage points over Logit, but its log-loss is poor at 0.8129, meaning its predicted probabilities are not well calibrated. The Random Forest has accuracy 82.33%, log-loss 0.3841, Brier score 0.1222, and ROC-AUC 0.8871. Therefore, the machine learning models outperform the traditional Logit: the tree is best for hard classification, while the forest is best for probability quality and ranking.

## AI Prompt for This Code Cell

Inspect feature importance for the tree-based models and compare it with the average linear coefficients from the Logit model.

In [5]:
tree_importance = pd.Series(
    decision_tree.feature_importances_,
    index=features,
    name="decision_tree_importance",
).sort_values(ascending=False)

forest_importance = pd.Series(
    random_forest.feature_importances_,
    index=features,
    name="random_forest_importance",
).sort_values(ascending=False)

importance_table = pd.concat([tree_importance, forest_importance], axis=1).fillna(0)
importance_table.round(4)

,decision_tree_importance,random_forest_importance
cost_diff,0.4544,0.3508
time_diff,0.3066,0.3331
transfers,0.1910,0.1865
reliability_diff,0.0480,0.1150
income_high,0.0000,0.0147


## Output Interpretation

The feature-importance table helps explain why the Random Forest performs well. In the Random Forest, `cost_diff` has the largest importance (0.3508), followed by `time_diff` (0.3331), `transfers` (0.1865), `reliability_diff` (0.1150), and `income_high` (0.0147). The Decision Tree also emphasizes `cost_diff` (0.4544) and `time_diff` (0.3066). This matches the data-generating rules: sharp changes occur when car is much faster or much cheaper. Unlike Logit coefficients, feature importance does not tell us whether a variable increases or decreases utility. It tells us how much the variable helped the tree-based model split the data for prediction.

## AI Prompt for This Code Cell

Summarize the lesson from this nonlinear example. Explain why machine learning performs better here and what limitation remains.

In [6]:
best_by_accuracy = comparison.sort_values("accuracy", ascending=False).iloc[0]
best_by_log_loss = comparison.sort_values("log_loss", ascending=True).iloc[0]

print(
    f"Best model by accuracy: {best_by_accuracy['model']} "
    f"({best_by_accuracy['accuracy']:.2%})"
)
print(
    f"Best model by log-loss: {best_by_log_loss['model']} "
    f"({best_by_log_loss['log_loss']:.4f})"
)

conclusion = (
    "In this nonlinear threshold-based example, the machine learning models "
    "outperform the traditional linear Logit. The Decision Tree has the best "
    "classification accuracy, while the Random Forest gives the best probability "
    "quality by log-loss. The traditional Logit remains easier to interpret, "
    "but its linear utility specification underfits the threshold and interaction "
    "patterns in the data."
)

print(conclusion)

Best model by accuracy: Decision Tree (82.67%)
Best model by log-loss: Random Forest (0.3841)
In this nonlinear threshold-based example, the machine learning models outperform the traditional linear Logit. The Decision Tree has the best classification accuracy, while the Random Forest gives the best probability quality by log-loss. The traditional Logit remains easier to interpret, but its linear utility specification underfits the threshold and interaction patterns in the data.


## Output Interpretation

The final output identifies the Decision Tree as the best model by accuracy (82.67%) and the Random Forest as the best model by log-loss (0.3841). This is the key contrast with Notebook 05. When the data-generating process is close to a linear utility model, traditional Logit can perform very well. When the data contain strong thresholds and interactions, machine learning models can predict better because they can partition the feature space. The limitation is interpretability: the machine learning models give better predictive results here, but their feature importances do not replace the behavioral clarity of Logit coefficients. The correct teaching conclusion is conditional, not universal: machine learning is better in this example because the data structure favors nonlinear rules.